In [1]:
!pip install wandb

In [2]:
import tensorflow as tf
import wandb

from wandb.keras import WandbMetricsLogger

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.preprocessing.image import ImageDataGenerator

2026-05-29 09:57:33.563251: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780048653.781367      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780048653.852468      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780048654.385720      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780048654.385764      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780048654.385768      58 computation_placer.cc:177] computation placer alr

ModuleNotFoundError: No module named 'wandb.keras'

In [4]:
from wandb.integration.keras import WandbMetricsLogger

In [5]:
import tensorflow as tf
import wandb

from wandb.integration.keras import WandbMetricsLogger

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization

from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kavyathokala753 (kavyathokala753-nit-jalandhar) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [28]:
train_path = '/kaggle/input/datasets/aryanpandey1109/inaturalist12k/Data/inaturalist_12K/train'

In [29]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.1,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [30]:
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

Found 9000 images belonging to 10 classes.


In [31]:
validation_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 999 images belonging to 10 classes.


In [32]:
def create_model(filters=32,
                 activation='relu',
                 dense_neurons=128,
                 dropout_rate=0.2,
                 batch_norm=True):

    model = Sequential()

    filter_list = [
        filters,
        filters*2,
        filters*4,
        filters*4,
        filters*8
    ]

    for i, f in enumerate(filter_list):

        if i == 0:

            model.add(
                Conv2D(
                    f,
                    (3,3),
                    padding='same',
                    input_shape=(224,224,3)
                )
            )

        else:

            model.add(
                Conv2D(
                    f,
                    (3,3),
                    padding='same'
                )
            )

        if batch_norm:
            model.add(BatchNormalization())

        model.add(Activation(activation))

        model.add(MaxPooling2D(pool_size=(2,2)))

        model.add(Dropout(dropout_rate))

    model.add(Flatten())

    model.add(
        Dense(
            dense_neurons,
            activation=activation
        )
    )

    model.add(Dropout(dropout_rate))

    model.add(Dense(10, activation='softmax'))

    return model

In [33]:
sweep_config = {

    'method': 'random',

    'metric': {
        'name': 'val_accuracy',
        'goal': 'maximize'
    },

    'parameters': {

        'filters': {
            'values': [32, 64]
        },

        'activation': {
            'values': ['relu', 'tanh']
        },

        'dense_neurons': {
            'values': [128, 256]
        },

        'dropout_rate': {
            'values': [0.2, 0.3]
        },

        'batch_norm': {
            'values': [True, False]
        },

        'batch_size': {
            'values': [32, 64]
        },

        'epochs': {
            'values': [5]
        }
    }
}

In [34]:
sweep_id = wandb.sweep(
    sweep_config,
    project='inaturalist-cnn'
)

Create sweep with ID: yvdyslkc
Sweep URL: https://wandb.ai/kavyathokala753-nit-jalandhar/inaturalist-cnn/sweeps/yvdyslkc


In [35]:
def train_model():

    wandb.init()

    config = wandb.config

    model = create_model(
        filters=config.filters,
        activation=config.activation,
        dense_neurons=config.dense_neurons,
        dropout_rate=config.dropout_rate,
        batch_norm=config.batch_norm
    )

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    train_generator.batch_size = config.batch_size

    model.fit(
        train_generator,
        validation_data=validation_generator,
        epochs=config.epochs,
        callbacks=[WandbMetricsLogger()]
    )

In [36]:
wandb.agent(
    sweep_id,
    function=train_model,
    count=10
)

wandb: Agent Starting Run: y648m4b3 with config:
wandb: 	activation: tanh
wandb: 	batch_norm: True
wandb: 	batch_size: 64
wandb: 	dense_neurons: 128
wandb: 	dropout_rate: 0.3
wandb: 	epochs: 5
wandb: 	filters: 64
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/5


I0000 00:00:1780049921.510420     207 service.cc:152] XLA service 0x79e2dc017250 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780049921.510485     207 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780049921.510489     207 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780049922.383012     207 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-29 10:18:53.444150: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 10:18:53.633278: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1780049946.892524     207 device_co

 18/141 ━━━━━━━━━━━━━━━━━━━━ 3:07 2s/step - accuracy: 0.1113 - loss: 4.1684

2026-05-29 10:19:41.892357: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 10:19:42.074529: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


...

 99/282 ━━━━━━━━━━━━━━━━━━━━ 1:47 587ms/step - accuracy: 0.0945 - loss: 2.5067

2026-05-29 10:53:42.758535: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 10:53:42.922245: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


162/282 ━━━━━━━━━━━━━━━━━━━━ 1:15 629ms/step - accuracy: 0.0969 - loss: 2.4414

wandb: Ctrl + C detected. Stopping sweep.


282/282 ━━━━━━━━━━━━━━━━━━━━ 212s 698ms/step - accuracy: 0.1033 - loss: 2.3166 - val_accuracy: 0.0991 - val_loss: 2.2978
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 175s 620ms/step - accuracy: 0.1369 - loss: 2.2703 - val_accuracy: 0.1281 - val_loss: 2.2732
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 180s 637ms/step - accuracy: 0.1503 - loss: 2.2462 - val_accuracy: 0.1491 - val_loss: 2.2471
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 182s 646ms/step - accuracy: 0.1839 - loss: 2.2073 - val_accuracy: 0.1882 - val_loss: 2.1992
Epoch 5/5
118/282 ━━━━━━━━━━━━━━━━━━━━ 1:36 590ms/step - accuracy: 0.1908 - loss: 2.1836

In [25]:
print(train_generator.class_indices)

{'train': 0, 'val': 1}
